# 05 — Cross-Encoder Reranker
**Phase:** Production Hardening — Day 6-7

**What this notebook does:**
- Adds a cross-encoder reranker after hybrid retrieval
- Retrieves top-10, reranks to top-3 with much higher precision
- Compares before/after reranking on the same queries

**Why rerank?**
Bi-encoders (sentence-transformers) embed query and chunk separately — fast but imprecise.
Cross-encoders read query + chunk together — slow but very precise.
Solution: retrieve many cheaply, rerank few precisely.

In [ ]:
!pip install sentence-transformers rank-bm25 -q

In [ ]:
import chromadb
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection("my_docs")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load all chunks for BM25
all_data   = collection.get(include=["documents"])
all_chunks = all_data["documents"]
all_ids    = all_data["ids"]
tokenised  = [c.lower().split() for c in all_chunks]
bm25       = BM25Okapi(tokenised)

print(f"Chunks: {len(all_chunks)}")
print("Loading cross-encoder reranker (downloads ~80MB first time)...")

# Cross-encoder: reads query + chunk together for precise scoring
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded!")

## Step 1 — Rebuild hybrid retrieval from notebook 04

In [ ]:
def vector_search(query, top_k=10):
    vec = embed_model.encode([query]).tolist()
    res = collection.query(query_embeddings=vec, n_results=top_k)
    return res["ids"][0], res["distances"][0], res["documents"][0]

def bm25_search(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [all_ids[i] for i in top_idx], [scores[i] for i in top_idx], [all_chunks[i] for i in top_idx]

def rrf_fusion(v_ids, b_ids, k=60):
    scores = {}
    for rank, cid in enumerate(v_ids):
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank + 1)
    for rank, cid in enumerate(b_ids):
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_retrieve(query, top_k=10):
    v_ids, _, v_chunks = vector_search(query, top_k)
    b_ids, _, b_chunks = bm25_search(query, top_k)
    fused = rrf_fusion(v_ids, b_ids)

    results = []
    id_to_chunk = {cid: chunk for cid, chunk in zip(v_ids + b_ids, v_chunks + b_chunks)}
    for cid, score in fused[:top_k]:
        if cid in id_to_chunk:
            results.append({"id": cid, "rrf_score": score, "text": id_to_chunk[cid]})
    return results

print("Hybrid retrieval ready")

## Step 2 — Reranking function
Cross-encoder scores each (query, chunk) pair together — much more precise.

In [ ]:
def rerank(query, candidates, top_k=3):
    """
    Takes a query and list of candidate chunks.
    Scores each (query, chunk) pair with cross-encoder.
    Returns top_k reranked results.
    """
    if not candidates:
        return []

    # Build pairs: [(query, chunk_text), ...]
    pairs = [(query, c["text"]) for c in candidates]

    # Cross-encoder scores each pair — reads both together
    scores = reranker.predict(pairs)

    # Attach score to each candidate
    for i, c in enumerate(candidates):
        c["rerank_score"] = float(scores[i])

    # Sort by rerank score descending
    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]


def retrieve_and_rerank(query, top_k=3):
    """
    Full pipeline: hybrid retrieves top-10, reranker picks top-3.
    This is the production-grade retrieval function.
    """
    # Step 1: Retrieve top-10 candidates cheaply
    candidates = hybrid_retrieve(query, top_k=10)

    # Step 2: Rerank precisely with cross-encoder
    final = rerank(query, candidates, top_k=top_k)

    return final

## Step 3 — Compare before and after reranking

In [ ]:
def compare_reranking(query):
    print(f"\n{'='*60}")
    print(f" Query: {query}")
    print(f"{'='*60}")

    # Before reranking — hybrid top-3
    before = hybrid_retrieve(query, top_k=3)
    print("\nBEFORE reranking (hybrid top-3):")
    for r in before:
        print(f"  {r['id']:15}  rrf={r['rrf_score']:.4f}")
        print(f"  {r['text'][:120]}...")

    # After reranking
    after = retrieve_and_rerank(query, top_k=3)
    print("\nAFTER reranking (cross-encoder top-3):")
    for r in after:
        print(f"  {r['id']:15}  rerank_score={r['rerank_score']:.4f}")
        print(f"  {r['text'][:120]}...")

    # Show rank changes
    before_order = [r['id'] for r in before]
    after_order  = [r['id'] for r in after]
    changed = [cid for i, cid in enumerate(after_order) if i < len(before_order) and cid != before_order[i]]
    print(f"\nRank changed for: {changed if changed else 'none — order stable'}")


# CHANGE these to match your document
compare_reranking("What is the main topic of this document?")
compare_reranking("specific technical term from your PDF")

## Step 4 — Final production retrieval function
Copy this into notebook 03 to replace `retrieve_chunks()`.

In [ ]:
def retrieve_chunks_production(query, top_k=3):
    """
    Production retrieval: hybrid search → cross-encoder rerank.
    Returns (chunks, ids, scores) — same interface as notebook 03.
    """
    results = retrieve_and_rerank(query, top_k=top_k)
    chunks = [r["text"]          for r in results]
    ids    = [r["id"]            for r in results]
    scores = [r["rerank_score"]  for r in results]
    return chunks, ids, scores

# Test
chunks, ids, scores = retrieve_chunks_production("What is this document about?")
print("Production retrieval:")
for cid, score in zip(ids, scores):
    print(f"  {cid}  rerank_score={score:.4f}")

## Key observations
- Cross-encoder scores are raw logits — higher = more relevant (no upper bound)
- Score > 5.0 = very strong match
- Score < 0.0 = probably irrelevant
- If before/after order is the same, hybrid was already accurate — reranker confirms
- If order changes significantly, cross-encoder caught a ranking error from bi-encoder

**Next:** `06_citation_hallucination_guard.ipynb` — enforce citations and build the confidence threshold system.